In [4]:
# load alice wonderful text data

with open("alice.txt","r",encoding="utf-8") as f:
  text=f.read()

start = text.find("CHAPTER I.")
text = text[start:]
print(text[:100])

CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race 


In [5]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [6]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts([text])

In [7]:
# total unique words in the dataset
len(tokenizer.word_index)
# tokenizer.word_counts

3513

In [8]:
#split data into inp and output per sequence
input_seq=[]
max_len=0
for sentence in text.split("\n"):
  tokenize_sentence=tokenizer.texts_to_sequences([sentence])[0]
  max_len=max(max_len,len(tokenize_sentence))

  for i in range(1,len(tokenize_sentence)):
    input_seq.append(tokenize_sentence[:i+1])


In [9]:
# Add extra zeros to fulfil size according to max size sequence
padded_inp=pad_sequences(input_seq,maxlen=max_len,padding="pre")

In [10]:
X=padded_inp[:,:-1]
y=padded_inp[:,-1]


In [11]:
# make catagories from y(ouput)
from tensorflow.keras.utils import to_categorical
y=to_categorical(y,num_classes=3514)
y.shape

(27974, 3514)

In [12]:
from tensorflow.keras.layers import Dense,Embedding,LSTM,GRU
from tensorflow.keras.models import Sequential

In [13]:
#build archtecture of model
from tensorflow.keras.layers import Input

model = Sequential([
    Input(shape=(18,)),
    Embedding(input_dim=3514, output_dim=100),
    GRU(128),
    Dense(3514, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 18, 100)        │       351,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 128)            │        88,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3514)           │       453,306 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 893,026 (3.41 MB)

 Trainable params: 893,026 (3.41 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
# compile model
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["accuracy"])

In [15]:
# train model
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    min_delta=0.001,
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

model.fit(X,y,epochs=100,callbacks=[early_stop])

Epoch 1/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - accuracy: 0.0750 - loss: 6.2087
Epoch 2/100
 26/875 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.1083 - loss: 5.5232

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1251 - loss: 5.3660
Epoch 3/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.1669 - loss: 4.8442
Epoch 4/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.1965 - loss: 4.4253
Epoch 5/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.2243 - loss: 4.0622
Epoch 6/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.2534 - loss: 3.7365
Epoch 7/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.2865 - loss: 3.4404
Epoch 8/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.3252 - loss: 3.1663
Epoch 9/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.3673 - loss: 2.9136
Epoch 10/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.4074 - loss: 2.6801
Epoch 11/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.4456 - loss: 2.4645
Epoch 12/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.4825 - loss: 2.2693
Epoch 13/100
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/st

In [16]:
# predict next word
text="It"
for i in range(10):
  token_text=tokenizer.texts_to_sequences([text])[0]
  padded_token_text=pad_sequences([token_text],maxlen=18,padding="pre")
  pos=np.argmax(model.predict(padded_token_text,verbose=0))
  for word,index in tokenizer.word_index.items():
    if index==pos:
      text=text+" "+word
      print(text)





It was
It was all
It was all very
It was all very well
It was all very well to
It was all very well to say
It was all very well to say “drink
It was all very well to say “drink me
It was all very well to say “drink me ”
It was all very well to say “drink me ” but
